# 04 · Red-noise / fGn extension / 紅雜訊與 fGn 擴展

Show what happens when the background is persistent (H > 0.5), why the white-noise baseline misfires, and how to fix it with Monte Carlo + fGn.

示範當背景為持續型雜訊時，白雜訊基準線會失效，以及如何用 Monte Carlo + fGn 修正。

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path('..').resolve()))

import numpy as np
import matplotlib.pyplot as plt
from PyEMD import EMD

from emdsig import (
    compute_et, chi2_confidence_bounds, monte_carlo_bounds,
    calibrate_intercept, generate_fgn, red_noise_slope,
)
rng = np.random.default_rng(7)

## 1. Generate fGn with H=0.8 / 產生 H=0.8 的 fGn

In [ ]:
N = 2048
H = 0.8
bg = generate_fgn(N, hurst=H, rng=rng)
signal = 1.5 * np.sin(2 * np.pi * np.arange(N) / 128)   # period-128 sine
x = signal + bg

fig, ax = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
ax[0].plot(bg, lw=0.6); ax[0].set_title(f'fGn background, H={H}')
ax[1].plot(x, lw=0.6);  ax[1].set_title('signal + fGn')
plt.tight_layout(); plt.show()

## 2. What happens if we use the *white* baseline? / 誤用白雜訊基準線會怎樣？

In [ ]:
imfs = EMD().emd(x)
E, T = compute_et(imfs)
valid = ~np.isnan(T)
lnE, lnT = np.log(E[valid]), np.log(T[valid])

c_white = calibrate_intercept(lnT[0], lnE[0])
grid = np.linspace(lnT.min() - 0.5, lnT.max() + 0.5, 200)
_, hi_white = chi2_confidence_bounds(grid, N=N, alpha=0.05, baseline_intercept=c_white)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(lnT, lnE, color='crimson', zorder=5, label='IMFs')
ax.plot(grid, -grid + c_white, 'b--', label='white-noise baseline (slope -1)')
ax.plot(grid, hi_white, 'b:', label='white 95% upper (wrong!)')
ax.set_xlabel('ln(T)'); ax.set_ylabel('ln(E)')
ax.set_title('Mis-applying the white-noise test to fGn data')
ax.legend(); ax.grid(True, alpha=0.3)
plt.show()
print('Notice how low-frequency IMFs are wrongly flagged as significant.')

## 3. Correct approach: Monte Carlo with fGn / 正確作法：Monte Carlo + fGn

In [ ]:
grid_mc, lo_mc, hi_mc = monte_carlo_bounds(
    N=N, n_trials=80, noise='fgn', hurst=H,
    emd_method='emd', seed=7, alpha=0.05,
)

order = np.argsort(grid_mc)
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(lnT, lnE, color='crimson', zorder=5, label='IMFs of real mixture')
ax.fill_between(grid_mc[order], lo_mc[order], hi_mc[order],
                color='steelblue', alpha=0.3, label=f'fGn 95% CI (H={H})')
slope = red_noise_slope(H)
ax.plot(grid_mc[order], slope * grid_mc[order] + (lo_mc[order][0] - slope * grid_mc[order][0]),
        'g--', alpha=0.5, label=f'fGn baseline (slope {slope:.1f})')
ax.set_xlabel('ln(T)'); ax.set_ylabel('ln(E)')
ax.set_title(f'Correct test with fGn background, H={H}')
ax.legend(); ax.grid(True, alpha=0.3)
plt.show()

## 4. Takeaway / 總結

- Injected sine at period 128 → ln(T) ≈ 4.85, should still sit above the fGn band.
- Low-frequency IMFs that looked 'significant' under the white-noise test now correctly sit inside the fGn band.

在注入的 128 週期正弦（ln(T) ≈ 4.85）之外，低頻 IMF 在白雜訊假設下會被誤判；換成 fGn 基準線之後，它們正確落回信賴帶之內。